In [1]:
import os
import pandas as pd
import numpy as np
import torch
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
from PIL import Image
import pydicom
from tqdm import tqdm
from sklearn.metrics import accuracy_score, recall_score, f1_score

In [2]:
class VinDrMammoDataset(Dataset):
    def __init__(self, csv_file, img_dir, view_filter='MLO', split_filter='training', transform=None, is_dicom=True):
        # 1. Carrega o CSV
        self.df = pd.read_csv(csv_file)
        
        # 2. Filtra a projeção (ex: apenas MLO)
        if view_filter:
            self.df = self.df[self.df['view_position'] == view_filter]
            
        # 3. Filtra a divisão de dados (ex: apenas 'training' ou 'test')
        if split_filter:
            self.df = self.df[self.df['split'] == split_filter]
            
        # 4. Remove linhas sem classificação e reseta o índice
        self.df = self.df.dropna(subset=['breast_birads']).reset_index(drop=True)
        
        self.img_dir = img_dir
        self.transform = transform
        self.is_dicom = is_dicom

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        study_id = row['study_id']
        image_id = row['image_id']
        
        # Caminho: images/study_id/image_id.dicom
        ext = ".dicom" if self.is_dicom else ".png"
        img_path = os.path.join(self.img_dir, study_id, f"{image_id}{ext}")
        
        # Carregamento da imagem DICOM
        if self.is_dicom:
            dcm = pydicom.dcmread(img_path)
            img_array = dcm.pixel_array.astype(float)
            
            # Normalização simples do DICOM
            img_array = (np.maximum(img_array, 0) / img_array.max()) * 255.0
            img_array = np.uint8(img_array)
            image = Image.fromarray(img_array).convert("RGB")
        else:
            image = Image.open(img_path).convert("RGB")
            
        # 5. Tratamento do texto "BI-RADS X" vindo do CSV
        birads_str = str(row['breast_birads'])
        # Pega apenas os números da string (ex: 'BI-RADS 2' vira 2)
        birads_num = int(''.join(filter(str.isdigit, birads_str)))
        
        # Subtrai 1 para o PyTorch (classes precisam começar em 0)
        y_label = torch.tensor(birads_num - 1, dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, y_label

In [3]:
IMG_SIZE = 512

# 1. Pipeline de Treino (Com Data Augmentation)
# O Data Augmentation ajuda a evitar overfitting criando variações das imagens
transformacoes_com_data_augmentation = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5), # Espelha horizontalmente 50% das imagens
    transforms.RandomRotation(degrees=15),  # Aplica rotações aleatórias de até 15 graus
    transforms.ToTensor(),                  # Converte a imagem PIL/Numpy para Tensor do PyTorch
    # Normalização padrão do ImageNet (Crucial para Transfer Learning com a DenseNet)
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

# 2. Pipeline de Teste/Validação (Apenas preparo, sem alterações)
# Nunca aplicamos rotações ou espelhamentos nos dados que usaremos para avaliar o modelo!
transformacoes_apenas_redimensionamento = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
# Seu CSV do "Arquivo 1"
csv_path = "../dataset/vindr-mammo/breast-level_annotations.csv"
images_path = "../dataset/vindr-mammo/images"

In [5]:
# Cria o conjunto de treinamento
dataset_treino = VinDrMammoDataset(
    csv_file=csv_path, 
    img_dir=images_path,
    view_filter='MLO', 
    split_filter='training', # <- Pega só os dados de treino
    is_dicom=True,
    transform=transformacoes_com_data_augmentation
)

# Cria o conjunto de validação/teste
dataset_teste = VinDrMammoDataset(
    csv_file=csv_path, 
    img_dir=images_path,
    view_filter='MLO', 
    split_filter='test',     # <- Pega só os dados de teste (verifique o nome exato no seu CSV)
    is_dicom=True,
    transform=transformacoes_apenas_redimensionamento # Sem augmentation no teste!
)

In [6]:
TAMANHO_LOTE = 16

dataloader_treino = DataLoader(
    dataset=dataset_treino, 
    batch_size=TAMANHO_LOTE, 
    shuffle=True, # Embaralha as imagens no treino
    num_workers=2 # Usa múltiplos núcleos do processador para carregar imagens mais rápido
)

dataloader_teste = DataLoader(
    dataset=dataset_teste, 
    batch_size=TAMANHO_LOTE, 
    shuffle=False, # Não precisamos embaralhar na validação
    num_workers=2
)

In [8]:
# 1. Configura o dispositivo (GPU se disponível, senão CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Treinando em: {device}")

# 2. Carrega a DenseNet com Transfer Learning
modelo = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
num_features = modelo.classifier.in_features
modelo.classifier = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(num_features, 5)
) # 5 classes BI-RADS
modelo = modelo.to(device)

# 3. Função de Perda e Otimizador
# Usamos CrossEntropyLoss para multiclasse. 
# O Adam é um ótimo otimizador para começar (Learning Rate padrão de 0.001)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(modelo.parameters(), lr=0.0001) # LR um pouco menor para Transfer Learning

Treinando em: cuda


In [9]:
# 2. CALCULA OS PESOS (Insira aqui)
# Contamos as ocorrências de cada classe no CSV carregado
contagem_classes = dataset_treino.df['breast_birads'].value_counts().sort_index()
contagens = [contagem_classes.get(f'BI-RADS {i}', 0) for i in range(1, 6)]
total_amostras = sum(contagens)

# Cálculo do peso: inverso da frequência
n_classes = 5
pesos = [total_amostras / (n_classes * c) if c > 0 else 0 for c in contagens]
pesos_tensor = torch.FloatTensor(pesos).to(device)

# 3. DEFINE A FUNÇÃO DE PERDA COM OS PESOS
criterion = nn.CrossEntropyLoss(weight=pesos_tensor)

# 4. DEFINE O OTIMIZADOR
optimizer = optim.Adam(modelo.parameters(), lr=0.0001)

In [10]:
EPOCHS = 10

for epoch in range(EPOCHS):
    print(f"\nÉpoca {epoch+1}/{EPOCHS}")
    print("-" * 20)
    
    modelo.train() 
    running_loss = 0.0
    
    # Listas para guardar todas as previsões e gabaritos da época
    todas_previsoes = []
    todos_labels = []
    
    loop_treino = tqdm(dataloader_treino, desc="Treinando", leave=True)
    
    for imagens, labels in loop_treino:
        imagens = imagens.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        outputs = modelo(imagens)
        loss = criterion(outputs, labels)
        
        # Backward e Otimização
        loss.backward()
        optimizer.step()
        
        # Coletando a classe com a maior probabilidade (a previsão final da rede)
        _, preds = torch.max(outputs, 1)
        
        # Salvamos as previsões e labels convertendo de volta para a CPU
        todas_previsoes.extend(preds.cpu().numpy())
        todos_labels.extend(labels.cpu().numpy())
        
        running_loss += loss.item() * imagens.size(0)
        loop_treino.set_postfix(loss=loss.item())
        
    # --- CÁLCULO DAS MÉTRICAS AO FINAL DA ÉPOCA ---
    epoch_loss = running_loss / len(dataset_treino)
    
    # average='macro' calcula a métrica para cada classe e tira a média (trata todas com o mesmo peso)
    epoch_acc = accuracy_score(todos_labels, todas_previsoes)
    epoch_recall = recall_score(todos_labels, todas_previsoes, average='macro', zero_division=0)
    epoch_f1 = f1_score(todos_labels, todas_previsoes, average='macro', zero_division=0)
    
    print(f"Loss: {epoch_loss:.4f} | Acurácia: {epoch_acc:.4f} | Recall: {epoch_recall:.4f} | F1-Score: {epoch_f1:.4f}")

print("\nTreinamento concluído!")


Época 1/10
--------------------


Treinando: 100%|██████████| 500/500 [08:29<00:00,  1.02s/it, loss=2.26]


Loss: 1.5873 | Acurácia: 0.3750 | Recall: 0.2229 | F1-Score: 0.1977

Época 2/10
--------------------


Treinando: 100%|██████████| 500/500 [08:27<00:00,  1.02s/it, loss=1.49]


Loss: 1.5686 | Acurácia: 0.4071 | Recall: 0.2277 | F1-Score: 0.2096

Época 3/10
--------------------


...

Loss: 1.5354 | Acurácia: 0.4191 | Recall: 0.2482 | F1-Score: 0.2242

Época 9/10
--------------------


Treinando: 100%|██████████| 500/500 [08:24<00:00,  1.01s/it, loss=1.44]


Loss: 1.5374 | Acurácia: 0.4057 | Recall: 0.2613 | F1-Score: 0.2196

Época 10/10
--------------------


Treinando: 100%|██████████| 500/500 [08:24<00:00,  1.01s/it, loss=2.59]

Loss: 1.5172 | Acurácia: 0.4083 | Recall: 0.2492 | F1-Score: 0.2156

Treinamento concluído!


In [11]:
print("Configurando a rede para Fine-Tuning...")

# 1. Descongelamento Seletivo
# Vamos iterar por todas as camadas da sua DenseNet
for name, param in modelo.named_parameters():
    # Descongela o último bloco (denseblock4), a normalização final (norm5) e o classificador
    if "features.denseblock4" in name or "features.norm5" in name or "classifier" in name:
        param.requires_grad = True
        print(f"Camada DESCONGELADA para treino: {name}")
    else:
        # Garante que os blocos 1, 2 e 3 continuem congelados
        param.requires_grad = False

# 2. Otimizador com Taxas de Aprendizado (Learning Rates) Específicas
# REGRA DE OURO DO FINE-TUNING: A taxa de aprendizado das camadas profundas 
# deve ser pelo menos 10x menor que a do classificador. Se for muito alta, 
# você "destrói" o conhecimento prévio da rede.

optimizer = optim.Adam([
    # Taxa minúscula (1e-5) para ajustar suavemente os detectores de textura médica
    {'params': modelo.features.denseblock4.parameters(), 'lr': 1e-5}, 
    {'params': modelo.features.norm5.parameters(), 'lr': 1e-5},
    
    # Taxa normal (1e-4) para a camada que toma a decisão final das 5 classes
    {'params': modelo.classifier.parameters(), 'lr': 1e-4}
], weight_decay=1e-4) # weight_decay ajuda a evitar o overfitting

print("\nOtimizador pronto com múltiplas Learning Rates!")

Configurando a rede para Fine-Tuning...
Camada DESCONGELADA para treino: features.denseblock4.denselayer1.norm1.weight
Camada DESCONGELADA para treino: features.denseblock4.denselayer1.norm1.bias
Camada DESCONGELADA para treino: features.denseblock4.denselayer1.conv1.weight
Camada DESCONGELADA para treino: features.denseblock4.denselayer1.norm2.weight
Camada DESCONGELADA para treino: features.denseblock4.denselayer1.norm2.bias
Camada DESCONGELADA para treino: features.denseblock4.denselayer1.conv2.weight
Camada DESCONGELADA para treino: features.denseblock4.denselayer2.norm1.weight
Camada DESCONGELADA para treino: features.denseblock4.denselayer2.norm1.bias
Camada DESCONGELADA para treino: features.denseblock4.denselayer2.conv1.weight
Camada DESCONGELADA para treino: features.denseblock4.denselayer2.norm2.weight
Camada DESCONGELADA para treino: features.denseblock4.denselayer2.norm2.bias
Camada DESCONGELADA para treino: features.denseblock4.denselayer2.conv2.weight
Camada DESCONGELADA 

In [12]:
EPOCHS = 20 # Para Fine-Tuning, 20 épocas é um bom começo
melhor_f1_val = 0.0 # Variável para rastrear o melhor modelo
caminho_salvamento = "melhor_densenet_vindr.pth"

for epoch in range(EPOCHS):
    print(f"\n{'='*30}")
    print(f"ÉPOCA {epoch+1}/{EPOCHS}")
    print(f"{'='*30}")
    
    # -----------------------------------------
    # FASE 1: TREINAMENTO
    # -----------------------------------------
    modelo.train()
    running_loss_treino = 0.0
    preds_treino, labels_treino = [], []
    
    loop_treino = tqdm(dataloader_treino, desc="Treino", leave=True)
    for imagens, labels in loop_treino:
        imagens, labels = imagens.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = modelo(imagens)
        loss = criterion(outputs, labels) # Usando aquele criterion com os pesos!
        
        loss.backward()
        optimizer.step()
        
        _, preds = torch.max(outputs, 1)
        preds_treino.extend(preds.cpu().numpy())
        labels_treino.extend(labels.cpu().numpy())
        running_loss_treino += loss.item() * imagens.size(0)
        
    loss_treino = running_loss_treino / len(dataset_treino)
    f1_treino = f1_score(labels_treino, preds_treino, average='macro', zero_division=0)
    
    # -----------------------------------------
    # FASE 2: VALIDAÇÃO (O Teste Real)
    # -----------------------------------------
    modelo.eval() # Desliga dropout e congela os pesos para teste
    running_loss_val = 0.0
    preds_val, labels_val = [], []
    
    # with torch.no_grad() desliga o cálculo de gradientes (economiza MUITA memória)
    with torch.no_grad():
        loop_val = tqdm(dataloader_teste, desc="Validação", leave=True)
        for imagens, labels in loop_val:
            imagens, labels = imagens.to(device), labels.to(device)
            
            outputs = modelo(imagens)
            loss = criterion(outputs, labels)
            
            _, preds = torch.max(outputs, 1)
            preds_val.extend(preds.cpu().numpy())
            labels_val.extend(labels.cpu().numpy())
            running_loss_val += loss.item() * imagens.size(0)
            
    loss_val = running_loss_val / len(dataset_teste)
    f1_val = f1_score(labels_val, preds_val, average='macro', zero_division=0)
    recall_val = recall_score(labels_val, preds_val, average='macro', zero_division=0)
    
    # -----------------------------------------
    # RESULTADOS E SALVAMENTO AUTOMÁTICO
    # -----------------------------------------
    print(f"\nResumo da Época {epoch+1}:")
    print(f"Treino -> Loss: {loss_treino:.4f} | F1: {f1_treino:.4f}")
    print(f"Valid  -> Loss: {loss_val:.4f} | F1: {f1_val:.4f} | Recall: {recall_val:.4f}")
    
    # Se o modelo foi melhor na validação do que nas épocas anteriores, nós o salvamos!
    if f1_val > melhor_f1_val:
        print(f"⭐ Novo recorde! F1-Score subiu de {melhor_f1_val:.4f} para {f1_val:.4f}. Salvando modelo...")
        #torch.save(modelo.state_dict(), caminho_salvamento)
        melhor_f1_val = f1_val

print("\nTreinamento Finalizado! O melhor modelo está salvo como:", caminho_salvamento)


ÉPOCA 1/20


Validação: 100%|██████████| 125/125 [02:01<00:00,  1.03it/s]



Resumo da Época 1:
Treino -> Loss: 1.5275 | F1: 0.2210
Valid  -> Loss: 1.4981 | F1: 0.2368 | Recall: 0.2953
⭐ Novo recorde! F1-Score subiu de 0.0000 para 0.2368. Salvando modelo...

ÉPOCA 2/20


Validação: 100%|██████████| 125/125 [02:02<00:00,  1.02it/s]



Resumo da Época 2:
Treino -> Loss: 1.5091 | F1: 0.2163
Valid  -> Loss: 1.4994 | F1: 0.2419 | Recall: 0.2955
⭐ Novo recorde! F1-Score subiu de 0.2368 para 0.2419. Salvando modelo...

ÉPOCA 3/20


...


Resumo da Época 6:
Treino -> Loss: 1.5064 | F1: 0.2161
Valid  -> Loss: 1.5043 | F1: 0.2429 | Recall: 0.2986
⭐ Novo recorde! F1-Score subiu de 0.2422 para 0.2429. Salvando modelo...

ÉPOCA 7/20


Validação: 100%|██████████| 125/125 [02:01<00:00,  1.03it/s]



Resumo da Época 7:
Treino -> Loss: 1.5070 | F1: 0.2199
Valid  -> Loss: 1.5028 | F1: 0.2062 | Recall: 0.2888

ÉPOCA 8/20


Treino:  56%|█████▌    | 278/500 [04:40<03:44,  1.01s/it]


KeyboardInterrupt: 